In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Loading the dataset

In [ ]:
df = pd.read_csv('ProjectData(in).csv')
df.head()

The shape of the dataset

In [ ]:
df.shape

Import neccessary libraries

Check for missing values

In [ ]:
print(df.isnull().sum())

Check for Duplicated values

In [ ]:
df.duplicated().sum()

In [ ]:
#Data preparation for income bracket
#Bins: 0-50k (low), 50k -100K(Medium), 100k+(high)

bins = [0, 50000, 100000, np.inf]
labels = ['Low (<= $50k)', 'Medium ($50k - $100k)', 'High (> $100k)']
df['Income Bracket'] = pd.cut(df['Income'], bins=bins, labels=labels, right=True)
df.head()

In [ ]:
#Combined 'Marital_Gender' column for easier grouping (feature combination)
df['Marital_Gender'] = df['Marital Status'] + ' ' + df['Gender']
df.head()

In [ ]:
#create'Education_Comparison' to group College vs High School
# Treating Associate's, Bachelor's, Master's, Doctorate as "College/Higher"
def classify_education(edu):
    if edu == "High School Diploma":
        return "High School Diploma"
    else:
        return "College/Higher"

df['Education_Comparison'] = df['Education'].apply(classify_education)
df.head()

In [ ]:
# Set global plot style
plt.style.use('ggplot')

# --- 3. Plotting Demographic Distributions ---
fig1, axes1 = plt.subplots(2, 2, figsize=(15, 12))
fig1.suptitle('Demographic Distribution of Dataset', fontsize=18, y=1.02)

# Age Distribution
sns.histplot(df['Age'], kde=True, ax=axes1[0, 0], color='skyblue')
axes1[0, 0].set_title('Age Distribution')
axes1[0, 0].set_ylabel('Count')

# Gender Distribution
gender_counts = df['Gender'].value_counts()
sns.barplot(x=gender_counts.index, y=gender_counts.values, hue=gender_counts.index, legend=False, dodge=False, ax=axes1[0, 1], palette='pastel')
axes1[0, 1].set_title('Gender Distribution')
axes1[0, 1].set_ylabel('Count')

# Income Bracket Distribution
income_counts = df['Income Bracket'].value_counts().reindex(labels)
sns.barplot(x=income_counts.index, y=income_counts.values,hue=income_counts.index, legend=False, dodge=False,ax=axes1[1, 0], palette='viridis')
axes1[1, 0].set_title('Income Bracket Distribution')
axes1[1, 0].set_ylabel('Count')
axes1[1, 0].tick_params(axis='x', rotation=15)

# Marital Status Distribution
marital_counts = df['Marital Status'].value_counts()
sns.barplot(x=marital_counts.index, y=marital_counts.values, hue=marital_counts.index, legend=False, dodge=False, # Fix for warning
            ax=axes1[1, 1], palette='Set2')
axes1[1, 1].set_title('Marital Status Distribution')
axes1[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

Credit Rating:: $Excellent$: 781-850 $Good$: 661-780 $Fair$: 601-660 $Poor$: 500-600 $Very Poor$: 300-499 

In [ ]:
# --- 1. Create the new Credit Rating based on user-defined intervals ---
bins_credit = [300, 500, 601, 661, 781, 851] 
labels_credit = ['Very Poor', 'Poor', 'Fair', 'Good', 'Excellent']

# Create the categorical Credit_Rating column
df['Credit_Rating'] = pd.cut(
    df['Numeric credit score'], 
    bins=bins_credit, 
    labels=labels_credit, 
    right=False, 
    include_lowest=True
)

# --- 2. Create the Binary Target Variable (Credit Approved) ---
# 1 = Approved (Good or Excellent Score); 0 = Not Approved (Fair, Poor, Very Poor)
approved_ratings = ['Good', 'Excellent']
df['Credit_Approved'] = df['Credit_Rating'].apply(lambda x: 1 if x in approved_ratings else 0)

print(df['Credit_Approved'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

In [ ]:
for col in ['Home Ownership', 'Income Bracket', 'Marital_Gender', 'Education_Comparison']:
    df[col] = df[col].astype('object')

# Separate features (X) and target (y)
X = df.drop(columns=[
    'Credit_Approved', 'Credit Score', 'Numeric credit score', 'Income', 
    'Education', 'Marital Status', 'Gender', 'Credit_Rating'
])
y = df['Credit_Approved']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Explicitly define features for safety
categorical_features = ['Home Ownership', 'Income Bracket', 'Marital_Gender', 'Education_Comparison']
numerical_features = ['Age', 'Number of Children']

# Create the preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features), # Scale numerical features
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features) # Encode categorical features
    ],
    remainder='drop' # Drop unlisted columns for guaranteed clean output
)

# --- 1. Logistic Regression (Linear Model) Pipeline ---
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='liblinear', random_state=42))
])

# Train the Logistic Regression model
lr_pipeline.fit(X_train, y_train)

# Predict and evaluate
lr_y_pred = lr_pipeline.predict(X_test)
lr_y_proba = lr_pipeline.predict_proba(X_test)[:, 1]

# --- 2. Random Forest Classifier (Non-Linear Model) Pipeline ---
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train the Random Forest model
rf_pipeline.fit(X_train, y_train)

# Predict and evaluate
rf_y_pred = rf_pipeline.predict(X_test)
rf_y_proba = rf_pipeline.predict_proba(X_test)[:, 1]

# Print Evaluation Reports
print("--- Logistic Regression (Linear Model) Classification Performance ---")
print("ROC AUC Score:", roc_auc_score(y_test, lr_y_proba))
print("Classification Report:\n", classification_report(y_test, lr_y_pred))

print("\n--- Random Forest (Non-Linear Model) Classification Performance ---")
print("ROC AUC Score:", roc_auc_score(y_test, rf_y_proba))
print("Classification Report:\n", classification_report(y_test, rf_y_pred))

# --- 3. Visualization of Model Comparison (ROC Curve) ---
plt.figure(figsize=(8, 6))
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_y_proba)
plt.plot(lr_fpr, lr_tpr, label=f'Logistic Regression (AUC = {roc_auc_score(y_test, lr_y_proba):.2f})')
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_y_proba)
plt.plot(rf_fpr, rf_tpr, label=f'Random Forest (AUC = {roc_auc_score(y_test, rf_y_proba):.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess (AUC = 0.50)')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve Comparison for Credit Approval')
plt.legend()
plt.savefig('roc_curve_comparison.png')

In [ ]:
feature_names = rf_pipeline.named_steps['preprocessor'].get_feature_names_out()
importances = rf_pipeline.named_steps['classifier'].feature_importances_

# 2. Create a DataFrame to link importance scores to feature names
feature_importances = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

# Select top 10 features for visualization
top_features = feature_importances.head(10)

# 3. Plotting the Top Feature Importances
plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature',data=top_features, hue='feature', legend=False)
plt.title('Top 9 Feature Importances for Credit Approval (Random Forest)')
plt.xlabel('Relative Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('rf_feature_importance.png')

# Print the top 10 features for reference
print("\n--- Top 9 Feature Importances ---")
print(top_features)


In [ ]:
plt.figure(figsize=(10, 6)) # Create a new figure and a single axes object implicitly

# 1. Calculate the counts
home_counts = df['Home Ownership'].value_counts()

# 2. Create the bar plot on the current active axes (plt.gca() which is the one created by plt.figure)
ax = sns.barplot(
    x=home_counts.index,
    y=home_counts.values,
    # Setting hue and dodge=False is redundant if only one value_counts is used,
    # but we can keep the arguments for robustness.
    hue=home_counts.index,
    legend=False,
    dodge=False,
    palette='Set2'
)

# 3. Set the title and labels using the axes object 'ax'
ax.set_title('Home Ownership Distribution')
ax.set_ylabel('Count')

plt.tight_layout()
plt.show()